# 生物医学 AI Agent：BioEvidence Agent

**课程作业**：做一个能用 LLM + 外部生物医学工具回答研究问题的 agent，测试 3 题以上，分析一个失败案例，写成可执行的 Jupyter 报告。

**这个 agent 做的事情**：内核是通用的：拿到问题，自己决定读哪份技能说明、跑哪些查询脚本、查几轮，最后作答。工具只有 `read` / `write` / `edit` / `bash` 四个（pi 的默认四件），生物医学的判断与做法全在 `skills/` 目录里，内核不认任何一个领域。

**一句话结论**：证据来自两个一手数据库的实时查询——Open Targets Platform（疾病-靶点关联打分与逐条证据）与 NCBI PubMed（文献）；查询脚本把「模型记忆里的说法」换成「带 id 的记录」，让每句话都能被机器核对，未核验引用从「无法判断」变成「可计数、可归零」。这套核验不写在核心里，而是 `skills/evidence-discipline/scripts/verify.py` 一个脚本，拿本轮落盘的原始响应与答案比对。

**代码来源**：agent 内核不是随手写的循环，是照 pi（`@earendil-works/pi-agent-core`）的 `packages/agent/src/agent-loop.ts` 用 Python 移植的，移植思路对照冬瓜《Pi Agent 双轨教程》的 Python 源码精读篇（`第3章 Agent Loop` / `第5章 工具系统` / `第6章 消息系统`）。

**目录**：

| 路径 | 内容 |
|---|---|
| `agent/messages.py` | 内部消息类型与到 LLM 线格式的转换（两层消息） |
| `agent/tools.py` | 工具对象、JSON Schema 声明、参数校验 |
| `agent/loop.py` | Agent Loop（移植自 `agent-loop.ts` 的 `runLoop`） |
| `agent/agent.py` | 通用 `Agent` 类：上下文、轮次预算、中止、收束；不认任何领域 |
| `agent/tools_builtin.py` | 四个内置工具 `read` / `write` / `edit` / `bash`（照 pi `dist/core/tools/`） |
| `agent/skills.py` | 技能发现与提示词注入（照 pi `dist/core/skills.js`） |
| `agent/llm.py` | OpenAI 兼容网关接入层：SSE 逐块解析，正文与思考分开推送 |
| `agent/prompts.py` | 系统提示词，照 pi 的 `buildSystemPrompt` 分段拼装（preamble / 工具 / 规则 / 技能清单 / cwd），领域角色与输出格式不在这里 |
| `skills/` | 领域知识：每个技能一个目录，`SKILL.md` + `references/` + `scripts/`；输出格式与证据纪律在 `evidence-discipline` |
| `server.py` | Web 演示（唯一碰 web 与领域脚本的地方） |
| `report.ipynb` | 本报告 |


## 1. 任务定义

**研究问题类型**：基因与疾病的关联，以及这些关联的依据。选这个类型是因为它有两个好处：问题有明确的一手数据源，答案又能被判对错——关联强度是数字，依据是 PMID，不需要专家主观打分。

**测试用的四个问题**：

| 编号 | 问题 | 考什么 |
|---|---|---|
| Q1 | Which genes are most strongly associated with Alzheimer disease? | 疾病 → 基因方向，多轮检索与排行 |
| Q2 | Which genes are associated with ALS? | 疾病缩写歧义，需要自己消歧 |
| Q3 | Which diseases is APOE most strongly associated with? | 基因 → 疾病方向 |
| F1 | What are the major biomarkers for tuberculosis? | 失败案例探针：数据模型与问题不匹配 |

**agent 必须满足的三条**：

1. 多轮：自己决定调什么工具、调几次，不是一次 prompt 出答案。
2. 带证据：每条结论后面跟工具真正返回过的标识符。
3. 可核查：答案里出现的标识符，程序能判断它是不是本次运行中真出现过。

## 2. 原理：为什么必须是 agent loop

大模型有三种用法，差别只在「谁决定下一步」：

| 用法 | 谁决定下一步 | 生医场景的问题 |
|---|---|---|
| 直接调用 | 人（一次性 prompt） | 模型只能用训练时的记忆，答不了「这周数据库里 APOE 的分数是多少」，也无法给出 PMID |
| 工作流 | 人写死步骤 | 疾病名不确定时步骤就错了；问题换个方向（基因→疾病）流程全废 |
| Agent Loop | 模型自己 | 模型看到工具结果后决定下一步，可以在检索不到时改参数重查 |

本 agent 用的是第三种。循环的形状（`agent/loop.py` 的 `run_loop`，对应 pi `agent-loop.ts` 的 `runLoop`）：

```mermaid
flowchart LR
    A[用户问题] --> B[调模型<br/>带工具声明]
    B --> C{有 tool_call?}
    C -- 有 --> D[校验参数<br/>执行工具]
    D --> E[工具结果写回消息列表]
    E --> F{轮次预算用完?}
    F -- 否 --> B
    F -- 是 --> G[收束或停止]
    C -- 没有 --> H[最终答案]
```

从 pi 源码移植时保留的机制，以及为什么保留：

| pi 的机制 | 在 `agent-loop.ts` 的位置 | 保留的理由 |
|---|---|---|
| `stopReason` 硬停止（error / aborted） | `runLoop` 内 | 模型调用失败时不能再套一轮，否则会拿空答案继续烧额度 |
| `stopReason === "length"` 时整批工具调用判失败 | `failToolCallsFromTruncatedMessage` | 输出被截断时工具参数可能残缺，执行残缺参数等于编造参数 |
| 工具参数校验不过就不执行，改成错误结果回给模型 | `prepareToolCall` | 让模型自己看到错在哪并重发，而不是抛异常终止整个 agent |
| 工具抛异常也变成一条错误结果，循环继续 | `agent-loop.js` 的 `catch` | 文件不存在、命令超时都是正常情况，模型改参数就能接着跑；异常往上染会让一次手滑终止整题 |
| 一条消息里的多个工具调用并行执行、结果按原顺序回填 | `executeToolCallsParallel` | 三次网络查询并行省时间，顺序稳定便于复现 |
| 每轮 `turn_start` / `turn_end` / `tool_execution_start` / `tool_execution_end` 事件 | `emit` 调用点 | 事后能复盘 agent 到底做了什么，报告里的「用了哪些工具」直接由事件算出 |
| `shouldStopAfterTurn` 出口钩子 | 对应本实现的轮次预算 | 防无界循环；预算用完的行为正是下面的失败案例 |

去掉的部分：pi 的 steering / follow-up 两级消息队列、上下文压缩、扩展系统。后来加的多轮靠复用 `context.messages` 实现，没有压缩，长会话最终会撞模型窗口（见 §6.2 与 `docs/architecture.md` 第 7 节）。

四层架构、一次提问的逐步时序、多轮会话的三份状态与四种状态迁移写在 `docs/architecture.md`；本文只留判据、结果与失败分析。

In [1]:
# 报告放在交付目录里也要能 Restart Kernel & Run All：把项目根（含 agent/）加进 sys.path
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / 'agent' / 'loop.py').is_file():
        sys.path.insert(0, str(_p))
        break
else:
    raise RuntimeError('没找到项目根：上层目录里没有 agent/loop.py')

import inspect
import json
import subprocess
import time

from agent import Agent, BuiltinTools
from agent.config import gateway_config
from agent.skills import ROOT, load_skills

MODEL = 'auto/deepseek-v4.1-flash'
RUN_STARTED = time.time()
base_url, api_key = gateway_config()
print('模型网关:', base_url)
print('模型:', MODEL, '| 凭据:', '已配置' if api_key else '缺失')


def verify(res, since=RUN_STARTED):
    """核验交给技能脚本：答案落临时文件，脚本扫本轮原始响应后返回 JSON。"""
    path = ROOT / '.tmp' / 'answer.md'
    path.parent.mkdir(exist_ok=True)
    path.write_text(res.answer, encoding='utf-8')
    out = subprocess.run(
        ['python3', str(ROOT / 'skills/evidence-discipline/scripts/verify.py'),
         '--answer-file', str(path), '--since', str(int(since))],
        capture_output=True, text=True, timeout=120, check=False)
    res.verification = json.loads(out.stdout)
    return res


def show(res):
    print(f'提问：{res.question}')
    print(f'轮次：{res.turns}｜工具调用 {len(res.tool_calls)} 次｜事件 {len(res.events)} 条')
    print('工具序列：')
    for c in res.tool_calls:
        print('  -', c['tool'], json.dumps(c['args'], ensure_ascii=False))
    v = res.verification
    print(f"证据核验：引用 {len(v['cited'])} 个标识符，未核验 {len(v['unverified'])} 个，覆盖率 {v['coverage']}，台账 {v['ledger_size']} 条")
    if v['unverified']:
        print('  未核验：', v['unverified'])
    print('-' * 78)
    print(res.answer)

模型网关: http://172.20.0.1:3425/v1
模型: auto/deepseek-v4.1-flash | 凭据: 已配置


## 3. 移植后的循环：`run_loop`

下面这段就是 agent 的主体。它和教材第 3 章讲的结构一一对应：内层 `while` 一圈是一次 **turn**（一次模型调用 + 这批工具的执行），有工具调用才进下一圈，没有就是自然结束。

**先澄清代码的来路**：pi 的官方 SDK 只有 TypeScript，教材的「Python 版」是 markdown 里的改写片段（带 `...` 占位，没有可 import 的模块，其 README 自述是「用 Python 语法帮你看懂 TS 代码」），**照它没法直接跑**。`agent/` 是照 pi 的真实 TS 源码（`agent-loop.ts` / `types.ts` / `agent.ts`）逐个函数搬的，教材用来核对照应关系：`run_loop` 同名，保留它标为「内核」的部分，去掉它标为「叠加1 steering」与「叠加2 follow-up」的两层。

对照的具体版本：仓库 `github.com/earendil-works/pi` 的 `main` @ `05fd3fb`，读的是 `packages/agent/src/` 下的三个文件：`agent-loop.ts`（952 行）、`types.ts`（551 行）、`agent.ts`（623 行）。与已发布 tag `v1.5.2` 的差异 31 行，全在 abort 处理的位置（v1.5.2 在循环里 `if (signal?.aborted) break`，main 改成 `prepareToolCall` 开头提前返回 `Operation aborted`）；本实现没移植 `AbortSignal`，所以这处差异不影响移植结果。

### 3.1 一次提问的工作流

循环是机械结构，工作流是这个结构跑一次问题时的固定套路。以 Q1 为例（5 轮 8 次调用，序列取自 §6 的工具记录）：

| 步 | agent 的动作 | 为什么是这一步 |
|---|---|---|
| 1 | `read skills/evidence-discipline/SKILL.md` | 先拿到四段输出格式与「证据够不够」的判据，后面才知道何时停 |
| 2 | `read skills/open-targets-graphql/SKILL.md` | 拿到端点、字段名与脚本用法，不靠模型记忆拼 GraphQL |
| 3 | `bash ... ot.py search-disease "Alzheimer disease"` | 名称转 id，锁定 `MONDO_0004975`（缩写的消歧就在这一步） |
| 4 | `bash ... ot.py disease-targets MONDO_0004975 15` | 拿排行与分项打分，构成答案的结论段 |
| 5 | `bash ... ot.py disease-targets EFO_1001870 10` | 对晚发型条目再查一次，找反向证据（APOE 在那里排第一） |
| 6 | `bash ... ot.py evidence <disease> <gene>` ×2 | 对候选基因取逐条证据与 PMID，把「分数」补成「哪几篇文献」 |
| 7 | 收口作答，标识符逐个对上 §5 的台账 | 答案与证据链闭环，核验由脚本给出数字而不是自评 |

（序列里还有一次 `ls skills/open-targets-graphql/` 探目录，属开销不属步骤。）这套步骤不是人写死的流程：第 3 步锁定哪个 id、第 5 步要不要再查一遍、第 6 步查哪两个基因，都是模型看过上一步结果后决定的——这正是 §2 里「Agent Loop 与工作流」的区别，工作流由模型按当前结果现场拼出来。

In [2]:
from agent import loop

print(inspect.getsource(loop.run_loop))

async def run_loop(context: AgentContext, new_messages: list[AgentMessage], config: AgentLoopConfig, emit: Callable) -> None:
    has_more_tool_calls = True
    first_turn = True

    while has_more_tool_calls:
        if config.cancelled and config.cancelled():
            await emit_event(emit, {"type": "loop_stop", "reason": "用户中止"})
            return
        if not first_turn:
            await emit_event(emit, {"type": "turn_start"})
        first_turn = False

        message = await stream_assistant_response(context, config, emit)
        new_messages.append(message)

        # 硬停止：模型调用失败或中止，不再进入下一轮
        if message.stop_reason in ("error", "aborted"):
            await emit_event(emit, {"type": "turn_end", "message": message, "tool_results": []})
            return

        tool_calls = message.tool_calls
        tool_results: list[ToolResultMessage] = []
        if tool_calls:
            # stop_reason == "length" 说明输出被截断，参数可能不完整，一律不执行
            if message.stop_reason == 

## 4. 证据从哪来

两个数据源，都免密钥、都是机构维护的一手库：

| 数据源 | 接口 | 用了哪些字段 |
|---|---|---|
| Open Targets Platform | GraphQL，`https://api.platform.opentargets.org/api/v4/graphql` | `search`（疾病/靶点检索）、`disease.associatedTargets`（疾病→基因排行）、`target.associatedDiseases`（基因→疾病排行）、`score` 与 `datatypeScores`（关联强度按证据类型分解）、`evidences`（逐条证据：`datatypeId`、`datasourceId`、`score`、`literature` 即 PMID 列表） |
| NCBI PubMed | E-utilities `esearch` / `esummary` / `efetch` | PMID、年份、期刊、标题、摘要原文 |

两个源互补：Open Targets 给「有多强、靠什么类型的证据」，PubMed 给「具体哪篇文献」。

agent 不直接调这两个 API：请求写在技能自带的脚本里（`skills/open-targets-graphql/scripts/ot.py`、`skills/pubmed-eutils/scripts/pubmed.py`，只用标准库），agent 用 `bash` 跑它们。所以提升一个新的数据源只要加一个技能目录，不用改 agent 代码。

下面不经 agent，直接打两次接口，看看原始证据长什么样：

In [3]:
tools = {t.name: t for t in BuiltinTools(ROOT).as_tools()}

# 1. 先读技能正文（只占这一轮上下文）
skill = tools['read'].execute({'path': 'skills/open-targets-graphql/SKILL.md', 'limit': 18})
print(skill.content)

# 2. 跑技能自带脚本，真发一次请求；原始响应落到 .tmp/responses/
started = time.time()
result = tools['bash'].execute({'command': 'python3 skills/open-targets-graphql/scripts/ot.py disease-targets MONDO_0004975 5'})
print(result.content)

# 3. 造一段答案，用技能脚本核验：台账来自本轮写入的原始响应
tools['write'].execute({'path': '.tmp/probe-answer.md', 'content': 'APP (ENSG00000142192) score=0.8743；AD 对应 MONDO_0004975；另引用一个没查过的 PMID 41371438。'})
checked = tools['bash'].execute({'command': f'python3 skills/evidence-discipline/scripts/verify.py --answer-file .tmp/probe-answer.md --since {int(started)}'})
print(checked.content)

---
name: open-targets-graphql
description: 查 Open Targets（疾病-靶点关联打分与逐条证据）时读。含可直接跑的查询脚本与字段说明，覆盖名称转 id、疾病→靶点、靶点→疾病、一对关联的逐条证据。
---

# Open Targets Platform

端点 `https://api.platform.opentargets.org/api/v4/graphql`，POST GraphQL，免密钥。

## 怎么用

先跑脚本，别手写 GraphQL：

```bash
python3 skills/open-targets-graphql/scripts/ot.py search-disease "Alzheimer disease"
python3 skills/open-targets-graphql/scripts/ot.py search-target APOE
python3 skills/open-targets-graphql/scripts/ot.py disease-targets MONDO_0004975 10
python3 skills/open-targets-graphql/scripts/ot.py target-diseases ENSG00000130203 10


原始响应：/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/.tmp/responses/ot-disease-targets-1795083797881.json
Alzheimer disease (MONDO_0004975) 关联靶点共 13006 条
APP	ENSG00000142192	score=0.8743	literature=1.00, clinical=0.98, genetic_association=0.93
PSEN1	ENSG00000080815	score=0.8679	literature=0.99, genetic_association=0.95, genetic_literature=0.89
PSEN2	ENSG00000143801	score=0.8190	genetic_association=0.91, literature=0.89, genetic_literature=0.77
APOE	ENSG00000130203	score=0.7883	literature=1.00, genetic_association=0.91, genetic_literature=0.61
ABCA7	ENSG00000064687	score=0.7408	literature=0.97, genetic_association=0.86, genetic_literature=0.61

{"cited": ["PMID 41371438", "ENSG00000142192", "MONDO_0004975"], "unverified": ["PMID 41371438"], "verified_count": 2, "coverage": 0.667, "ledger_size": 6, "ledger": [{"identifier": "ENSG00000142192", "source": ".tmp/responses/ot-disease-targets-1795083797881.json", "snippet": "\"id\": \"ENSG00000142192\","}, {"identifier": "ENSG

### 4.1 工具清单与技能清单

工具只有四个，都是通用能力：`read` 读文件、`write` 写文件、`edit` 按原文精确替换一段、`bash` 跑命令（发请求、跑脚本、解析 JSON 都归它）。这是 pi 的默认四件（`DEFAULT_TOOL_NAMES`）；另外四个 `grep` / `find` / `ls` / `powershell` 没移植，前者在 `bash` 里直接跑，后者只在 Windows 上用。模型能看到的只有工具声明（名字、说明、参数 Schema），看不到实现。

领域知识不在工具里，在 `skills/` 目录：每个技能一个文件夹，`SKILL.md` 开头两行写明名字与「什么时候读我」，同时按规范校验（名字只允许小写字母、数字与短横线）；`references/` 放字段说明，`scripts/` 放能直接跑的查询脚本。启动时只把每个技能的名字与一句话摘要注入系统提示词，正文要模型自己用 `read` 拉开——它不读就不会知道具体字段名。

这是 pi 的做法：工具面固定，知识可加。代价是模型多花 2–4 次调用去读说明，好处是换数据源不需要改 agent 代码，也不会因为工具太多而在选择上摇摆。

In [4]:
for t in BuiltinTools(ROOT).as_tools():
    print(f"{t.name}({', '.join(t.parameters.get('required', []))})")
    print('   ', t.description.split('。')[0])
    print('    参数：', list(t.parameters['properties']))

print()
for s in load_skills().skills:
    print(f'{s.name}：{s.description}')
    print('   ', s.file_path.relative_to(ROOT))

read(path)
    读取文件内容
    参数： ['path', 'offset', 'limit']
write(path, content)
    写入文件
    参数： ['path', 'content']
edit(path, edits)
    按原文精确替换来改文件
    参数： ['path', 'edits']
bash(command)
    在当前目录执行 bash 命令，返回 stdout 与 stderr 合并后的结果；超限只留末尾，全文另存
    参数： ['command', 'timeout']

evidence-discipline：回答任何生物医学问题前必读。给出四段输出格式（结论/证据/缺口/方法）、「证据够不够」的判据与常见陷阱；核验由外部脚本执行，不需要自己跑。
    skills/evidence-discipline/SKILL.md
open-targets-graphql：查 Open Targets（疾病-靶点关联打分与逐条证据）时读。含可直接跑的查询脚本与字段说明，覆盖名称转 id、疾病→靶点、靶点→疾病、一对关联的逐条证据。
    skills/open-targets-graphql/SKILL.md
pubmed-eutils：查 PubMed 文献（检索式 → PMID → 标题摘要）时读。含可直接跑的检索与摘要脚本，以及 E-utilities 的参数与返回结构。
    skills/pubmed-eutils/SKILL.md


## 5. 证据台账与可核验性判据

做法：台账不写在 agent 里，而是技能自带的一个脚本 `skills/evidence-discipline/scripts/verify.py`。两个查询脚本把每次请求的原始响应写进 `.tmp/responses/`，核验脚本只算本轮（`--since` 之后写入）的文件，把出现过的 `PMID` / `ENSG` / `MONDO` 收成台账，再拿答案里出现的标识符去台账里查。

这个分界即代码结构里的那条线：`agent/` 只认工具与消息，不知道什么叫 PMID；生物医学的判据连同脚本留在技能里，换一个领域只换 `skills/` 目录，内核一行不动。代价是台账从「每次工具出口都扫一遍」变成「只扫落盘的原始响应」，没落盘的 bash 输出不进台账；换来的是内核里没有一行领域代码。

台账只能证明「这个 id 在本轮某次工具输出里出现过」，不能证明「数据库真的返回过它」——在 `bash` 里 `echo` 一个 id 同样会进台账。对课程作业这仍然是有效检查（它挡住了「凭记忆写 id」这类最常见的失败），但它不是密码学意义的证明。

判据是程序化的，不看模型自述：

| 指标 | 定义 |
|---|---|
| `cited` | 答案文本里被识别到的标识符（`ENSG\d{11}`、`MONDO/EFO_\d+`、`PMID: n`） |
| `unverified` | 引用了但台账里没有的标识符 —— 这是幻觉引用的直接证据 |
| `coverage` | 已核验引用数 ÷ 引用总数 |
| `ledger_size` | 台账记录数，反映这次实际取回了多少证据 |

所以「答案有没有证据支持」在这里是可计算的，而不是读一遍觉得挺像样。

## 6. 测试：三题

每题用同一个模型、同样 20 轮预算。记录问题、用到的工具、最终答案、核验结果。

In [5]:
Q1 = 'Which genes are most strongly associated with Alzheimer disease?'
t1 = time.time()
q1 = verify(await Agent(model=MODEL, max_turns=20, force_final_answer=True).ask(Q1), since=t1)
show(q1)

提问：Which genes are most strongly associated with Alzheimer disease?
轮次：5｜工具调用 8 次｜事件 541 条
工具序列：
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/evidence-discipline/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/open-targets-graphql/SKILL.md"}
  - bash {"command": "cat skills/open-targets-graphql/SKILL.md | head -60; echo ---; python3 skills/open-targets-graphql/scripts/ot.py search-disease \"Alzheimer disease\""}
  - bash {"command": "ls skills/open-targets-graphql/scripts/ skills/open-targets-graphql/references/"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/open-targets-graphql/scripts/ot.py disease-targets MONDO_0004975 15"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/open-targets-graphql/scripts/ot.py disease-targets EFO_1001870 10"}
  - bash {"command": "cd /

In [6]:
Q2 = 'Which genes are associated with ALS?'
t2 = time.time()
q2 = verify(await Agent(model=MODEL, max_turns=20, force_final_answer=True).ask(Q2), since=t2)
show(q2)

提问：Which genes are associated with ALS?
轮次：5｜工具调用 7 次｜事件 483 条
工具序列：
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/evidence-discipline/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/open-targets-graphql/SKILL.md"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/open-targets-graphql/scripts/ot.py search-disease \"amyotrophic lateral sclerosis\" 2>&1 | head -60"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && ls skills/open-targets-graphql/scripts/ && python3 skills/open-targets-graphql/scripts/ot.py search-disease \"ALS\" 2>&1 | head -60"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && ls && python3 skills/open-targets-graphql/scripts/ot.py disease-targets MONDO_0004976 20"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/b

In [7]:
Q3 = 'Which diseases is APOE most strongly associated with?'
t3 = time.time()
q3 = verify(await Agent(model=MODEL, max_turns=20, force_final_answer=True).ask(Q3), since=t3)
show(q3)

提问：Which diseases is APOE most strongly associated with?
轮次：11｜工具调用 15 次｜事件 955 条
工具序列：
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/evidence-discipline/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/open-targets-graphql/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/open-targets-graphql/SKILL.md"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/open-targets-graphql/scripts/ot.py search-target APOE 2>&1 | head -40"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/open-targets-graphql/scripts/ot.py target-diseases ENSG00000130203 25"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/open-targets-graphql/scripts/ot.py evidence MONDO_0018473 ENSG00000130203 5 

In [8]:
results = [('Q1', Q1, q1), ('Q2', Q2, q2), ('Q3', Q3, q3)]

print('| 题 | 轮次 | 工具调用 | 引用标识符 | 未核验 | 覆盖率 | 台账 |')
print('|---|---|---|---|---|---|---|')
for tag, q, res in results:
    v = res.verification
    print(f"| {tag} | {res.turns} | {len(res.tool_calls)} | {len(v['cited'])} | {len(v['unverified'])} | {v['coverage']} | {v['ledger_size']} |")
print()
for tag, q, res in results:
    print(tag, '用了：', ' → '.join(c['tool'] for c in res.tool_calls))

| 题 | 轮次 | 工具调用 | 引用标识符 | 未核验 | 覆盖率 | 台账 |
|---|---|---|---|---|---|---|
| Q1 | 5 | 8 | 27 | 0 | 1.0 | 53 |
| Q2 | 5 | 7 | 34 | 0 | 1.0 | 69 |
| Q3 | 11 | 15 | 17 | 0 | 1.0 | 60 |

Q1 用了： read → read → bash → bash → bash → bash → bash → bash
Q2 用了： read → read → bash → bash → bash → bash → bash
Q3 用了： read → read → read → bash → bash → bash → bash → bash → bash → bash → bash → bash → bash → bash → bash


### 6.1 我对三题答案的判断

判断口径：把 agent 给出的分数与 id 拿去 Open Targets 网页 / PubMed 原文核对一遍，看是否一致；再看结论与缺口两段有没有夸大。

| 题 | 判断 | 核对依据与观察 |
|---|---|---|
| Q1 阿尔茨海默病 | **正确** | 5 轮 8 次调用，27 个标识符全部可核验。整体排行 APP (ENSG00000142192) score=0.8743、PSEN1 0.8679、PSEN2 0.8190、APOE 0.7883、ABCA7 0.7408、SORL1 0.7147，与数据库排序一致。它还多查了晚发型条目 EFO_1001870（229 个靶点），给出 APOE 在那里排第一（0.6349）这条反向证据，并指出「按分项证据而非综合分排序会换人」（GRIN1 clinical=0.99、ACHE clinical=1.00）。缺口段点明这些分数只在同一个疾病内可比、不等于致病机制。 |
| Q2 ALS | **正确** | 5 轮 7 次调用，34 个标识符全部可核验。用全称 `amyotrophic lateral sclerosis` 一次锁定 MONDO_0004976（6329 个关联靶点），SOD1 0.8880 / TARDBP 0.8498 / FUS 0.8434 / UBQLN2 0.8344 / TBK1 0.8310 与数据库一致，连第 20 名 SIGMAR1 0.7525 都在。缺口段自己交代了名称检索命中多条 MONDO 条目（familial ALS MONDO_0005144、juvenile ALS MONDO_0017593），本答案只处理主条目——这条自我限制写对了。 |
| Q3 APOE | **正确** | 11 轮 15 次调用，17 个标识符全部可核验。首位 hyperlipoproteinemia type 3 (MONDO_0018473) score=0.8035、coronary artery disorder 0.7825、lipoprotein glomerulopathy 0.7591，AD 只排第四（0.6955）——反直觉的结论也照实写了，还补了一句「AD 在分数上不是第一，但文献体量最大」。轮次比另两题多，花在用 `bash` 里的 `python3 -c` 把 `.tmp/responses/` 下的原始 JSON 再解析一遍，确认字段没取错。 |

三题合计：78 个标识符，未核验 0，覆盖率 1.0。三处要看清：一、**覆盖率只说明引用在工具输出里出现过，不说明结论一定对**；二、轮次是 5/5/11，每题前两三次调用都花在读技能说明上，这是通用工具的固定开销；三、引用数 27/34/17 差别大，来自模型自查的松紧（Q2 把整份靶点排行都列了，Q3 只列六条），不是答案质量的差别。

### 6.2 多轮：接着上一轮的证据追问

前面每题都是新建一个 agent。真实使用是连续对话：agent 对象的 `context.messages` 不重建，上一轮的工具结果就留在上下文里，下一轮可以直接引用。这里测一组：先问阿尔茨海默病基因排行，再追加“这些基因里哪个证据最强”。

判据很直接：第二问应该**不再调工具**（答案所需的证据已在上下文里），而请求携带的消息数应该明显大于第一问。

In [9]:
multi = Agent(model=MODEL, max_turns=20, force_final_answer=True)
tm = time.time()
m1 = verify(await multi.ask(Q1), since=tm)
m2 = verify(await multi.ask('这些基因里哪个证据最强？只用一句话。'), since=tm)

for tag, res, calls in (
    ('第 1 问      ', m1, len(m1.tool_calls)),
    ('第 2 问（追问）', m2, len(m2.tool_calls)),
):
    size = next(e['message_count'] for e in res.events if e['type'] == 'request_start')
    print(f'{tag}：{res.turns} 轮｜调用工具 {calls} 次｜请求里带 {size} 条上下文消息')

print()
print(m2.answer)
print()
print('第 2 问核验：覆盖率', m2.verification['coverage'], '｜未核验', len(m2.verification['unverified']), '个')
print('两问共用的证据台账：', m2.verification['ledger_size'], '条')

第 1 问      ：8 轮｜调用工具 13 次｜请求里带 2 条上下文消息
第 2 问（追问）：1 轮｜调用工具 0 次｜请求里带 16 条上下文消息

**APP (ENSG00000142192)** 在仅覆盖 Alzheimer disease (MONDO_0004975) 的 disease→targets 排行里证据强度最高，整体 score=0.8743（literature 1.00、clinical 0.98、genetic_association 0.93），高于 PSEN1 (ENSG00000080815, 0.8679) 与 APOE (ENSG00000130203, 0.7883)。

第 2 问核验：覆盖率 1.0 ｜未核验 0 个
两问共用的证据台账： 60 条


第 2 问的数字说明两件事：工具调用 0 次，说明它没重查，而是从上下文里取上一轮 `ot.py disease-targets` 的输出；请求携带的消息数从 2 涨到 16，说明历史真的进了请求（第一问 8 轮，每轮的工具调用与结果都留在里面）。台账也是会话级的：两问共用 60 条，所以第 2 问引用的 APP、PSEN1、APOE 能被核到，覆盖率 1.0。

代价也要写明：上下文不复位，每轮请求就把前面的消息重发一次，轮数越多越贵。真实产品因此要做上下文压缩（pi 的 loop 里有这段，本实现没移植）。

## 7. 对照：不给工具会怎样

同一个模型、同样的四段输出要求（结论 / 证据 / 缺口 / 方法），唯一差别是不提供工具。这是作业里「tool use 是否提升可靠性」的直接实验。
先看一个具体差别：Q1 无工具时说「APOE 是关联最强的基因，尤其 ε4」，有工具时按 Open Targets 整体 score 说「APP 第一，APOE 第四」；两边对「最强」的定义不同（人群归因风险 vs 数据库加权分）。两边都在缺口段承认了这个歧义，但只有有工具的那边能给出可回查的分数与 PMID。

In [10]:
agent_baseline = Agent(model=MODEL)
baselines = []
for tag, q, _ in results:
    baselines.append((tag, q, verify(await agent_baseline.ask_without_tools(q), since=time.time())))

print('| 题 | 引用标识符 | 可核验 | 未核验 | 覆盖率 |')
print('|---|---|---|---|---|')
for tag, q, res in baselines:
    v = res.verification
    print(f"| {tag} | {len(v['cited'])} | {v['verified_count']} | {len(v['unverified'])} | {v['coverage']} |")
print()
for tag, q, res in baselines:
    print('=' * 20, tag, '无工具基线', '=' * 20)
    print(res.answer)
    print()

| 题 | 引用标识符 | 可核验 | 未核验 | 覆盖率 |
|---|---|---|---|---|
| Q1 | 0 | 0 | 0 | 0.0 |
| Q2 | 0 | 0 | 0 | 0.0 |
| Q3 | 0 | 0 | 0 | 0.0 |

==================== Q1 无工具基线 ====================
The strongest Alzheimer disease (AD) genes depend on whether you mean **late-onset AD risk** or **early-onset familial AD**:

### Strongest overall
1. **APOE** — by far the strongest common genetic risk factor for late-onset AD.  
   - **APOE ε4** increases risk in a dose-dependent way: heterozygotes have roughly **3–4×** risk; ε4/ε4 homozygotes roughly **10–15×** risk.  
   - **APOE ε2** is protective.  
   - APOE ε4 is neither necessary nor sufficient for AD.

### Rare, highly penetrant autosomal-dominant early-onset AD genes
These cause familial early-onset AD and are strongly causal, though rare:
- **PSEN1** — most common cause of autosomal-dominant early-onset AD
- **APP**
- **PSEN2** — less common

### Strong rare-variant risk genes
- **TREM2** — especially the **R47H** variant; odds ratio ~2–3
- **SOR

基线答案读起来很像样：术语对、结构齐、还给出了 OR 值区间和基因名排行。但它**一个标识符都没有**，也就是没有一条能被核对的引用——`cited = 0`，`verified = 0`。差别不在流畅度，在于「能不能查」。

更要紧的是这类答案会随时间与人群漂移：OR 区间、人群频率、哪个基因排第一，数据库一更新它就过期，而没人能从文本里看出它过期了。agent 的答案里每个分数都带 id 和查询时点，过期能发现。

所以工具带来的提升具体有三点：**引用可核验、数值可复算、缺口可定位**（agent 能说「PSEN2 的逐条证据这次没取」，基线只能笼统说「还有缺失遗传力」）。

## 8. 失败案例：预算耗尽时，答案退化成一句计划

**现象**：问「What are the major biomarkers for tuberculosis?」，8 轮预算下 agent 连查 16 次工具，预算用尽后答案是「我先阅读相关技能，然后查询 Open Targets 和 PubMed。」——一句开场白，没有结论、没有标识符，`coverage = 0`。下面是三次运行，把两个变量（预算、是否收束）拆开：

| 运行 | 预算 | 收束 | 轮次 | 工具调用 | 引用 | 覆盖率 |
|---|---|---|---|---|---|---|
| A | 8 轮 | 否 | 8 | 16 | 0 | 0.0 |
| B | 8 轮 | 是 | 9 | 16 | 27 | 1.0 |
| C | 20 轮 | 否 | 13 | 23 | 35 | 1.0 |

**为什么发生**（按重要性排）：

1. **预算耗尽时没有收束动作**。这是直接原因。循环照 pi 的 `shouldStopAfterTurn` 停下，直接返回已有消息；此时最后一条 assistant 消息是带 tool_calls 的那条，正文只有一句「我先阅读相关技能」，于是开工计划被当成最终答案。A 与 B 对比：同一个 8 轮预算，只多了一次不带工具的收束调用，答案就从一句话变成 27 个可核验标识符（尿里的 LAM、ESAT-6 / CFP-10 / MPT-64，血里的 RTP4、GBP1、C1q 等）。
2. **数据模型与问题不匹配，钱花在了错的检索上**。Open Targets 存的是「靶点–疾病关联」，不是「生物标志物」。A 的工具序列里两次 `pubmed.py search`（`tuberculosis AND biomarker AND (diagnosis OR diagnostic)`、`IP-10 CXCL10 tuberculosis diagnostic biomarker`）都在找临床标志物，而技能脚本只能给靶点排行；说明写清了返回什么，但没有一种工具能拒绝这类问题，于是模型没有「这个问题答不了」的停止信号。
3. **加预算只是有时管用**。C 把预算放到 20 轮，它查到 23 次、在 13 轮上自然停住，给出 35 个可核验标识符（覆盖率 1.0）——也就是说预算够时不需要收束。但这不能代替机制：预算只决定什么时候停，不决定停的时候最后一条消息长什么样。A 与 C 是同一个模型、同一个问题、同样的工具，区别只在剩余轮次够不够它把话说完；8 轮那条是废掉的 16 次调用，20 轮那条能用。收束解决的是「预算不够也得给答案」，而不是「多给预算总能答好」。
4. **规则里没有「够用就停」**。系统提示词写了「工具报错要重试一次」「拿不到就写缺口」，但没写「已有多少证据就够回答了」。模型倾向于继续补全，预算先到。

**修复**：在 `Agent` 加 `force_final_answer`，预算耗尽且最后一条 assistant 消息无正文时，追加一次不带工具的调用，要求「立刻按输出格式，用已有工具结果作答，拿不到的部分写进缺口」。它保证的是「在预算边界上一定给出可核验的答案」，而不是「保证成功」——答案覆盖面的问题仍在（见下一 cell）。

In [11]:
FAIL_Q = 'What are the major biomarkers for tuberculosis?'

ta = time.time()
broken = verify(await Agent(model=MODEL, max_turns=8, force_final_answer=False).ask(FAIL_Q), since=ta)
print('=== 运行 A：8 轮预算，预算耗尽直接停 ===')
show(broken)

tb = time.time()
fixed = verify(await Agent(model=MODEL, max_turns=8, force_final_answer=True).ask(FAIL_Q), since=tb)
print('=== 运行 B：8 轮预算 + 预算耗尽时收束 ===')
show(fixed)

tc = time.time()
relaxed = verify(await Agent(model=MODEL, max_turns=20, force_final_answer=False).ask(FAIL_Q), since=tc)
print('=== 运行 C：20 轮预算，不加收束 ===')
show(relaxed)

print('| 运行 | 预算 | 收束 | 轮次 | 工具调用 | 引用 | 覆盖率 |')
print('|---|---|---|---|---|---|---|')
for tag, res, budget, force in [('A', broken, 8, False), ('B', fixed, 8, True), ('C', relaxed, 14, False)]:
    v = res.verification
    print(f"| {tag} | {budget} | {'是' if force else '否'} | {res.turns} | {len(res.tool_calls)} | {len(v['cited'])} | {v['coverage']} |")

=== 运行 A：8 轮预算，预算耗尽直接停 ===
提问：What are the major biomarkers for tuberculosis?
轮次：8｜工具调用 16 次｜事件 441 条
工具序列：
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/evidence-discipline/SKILL.md"}
  - bash {"command": "ls /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/*/"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/open-targets-graphql/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/pubmed-eutils/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/open-targets-graphql/SKILL.md"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/open-targets-graphql/scripts/ot.py search-disease \"tuberculosis\""}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/open-targets-graphql/

=== 运行 B：8 轮预算 + 预算耗尽时收束 ===
提问：What are the major biomarkers for tuberculosis?
轮次：9｜工具调用 16 次｜事件 719 条
工具序列：
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/evidence-discipline/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/pubmed-eutils/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/open-targets-graphql/SKILL.md"}
  - bash {"command": "ls /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/*/scripts/"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/open-targets-graphql/scripts/ot.py search-disease \"tuberculosis\" 2>&1 | head -40"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/pubmed-eutils/scripts/pubmed.py search \"tuberculosis biomarker diagnosis[Title]\" 5 2>&1 | head -40"}
  - bash {"comm

=== 运行 C：20 轮预算，不加收束 ===
提问：What are the major biomarkers for tuberculosis?
轮次：13｜工具调用 23 次｜事件 1735 条
工具序列：
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/evidence-discipline/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/pubmed-eutils/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/open-targets-graphql/SKILL.md"}
  - read {"path": "/home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118/skills/pubmed-eutils/SKILL.md"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/open-targets-graphql/scripts/ot.py search-disease \"tuberculosis\" 2>&1 | head -60"}
  - bash {"command": "cd /home/zys/.my/workspace/tasks/learning/biomedical-agent-20261118 && python3 skills/pubmed-eutils/scripts/pubmed.py search \"tuberculosis biomarker diagnosis review\" 5 2>&1 | head -40"}
  - bash {"

**修复后的结果怎么读**：`force_final_answer` 不是在补知识，它只解决「半成品被当成成品」这一个问题。运行 B 的答案把两类证据分开了：文献层面的临床标志物（胸腔积液里的 GZMA，90% / 91% 的敏感度与特异度，PMID 39641275；血里的 B2M + TXN + PRDX5 三蛋白面板 AUC 0.9847，PMID 41016070；血浆 IP-10 + RANTES，PMID 30029914），与 Open Targets 的宿主靶点关联（NR3C1 0.5952、LTF 0.5178、TLR2 0.4899、VDR 0.4135、B2M 0.3756）。它还主动写清了 GZMA 在 Open Targets 的疾病排行里查不到结核这一条——「这个数据源盖不到这类标志物」，才是修复带来的关键产出，而不是留一句开场白。

**仍然存在的问题**（这类问题的正确做法还没实现）：应该在第 1 轮就识别「问题要的是临床标志物，而手上只有靶点关联」，先只用 PubMed 综述作答并声明数据边界，而不是硬查十几次。这需要在技能正文里加一条能力边界说明，或让 agent 先做一次意图——数据源匹配判断。

**这个案例说明的事**：工具能让答案可核验，但不能扩大数据源的覆盖面；当问题落在数据模型之外时，agent 的失败形态不是「编一个答案」，而是「查个不停，最后连自己原本能答的部分都没答」。

## 9. 结论

**Q：这个生物医学 agent 怎么获得证据？**

四层：技能层放着两个查询脚本（`ot.py` 五个子命令、`pubmed.py` 两个子命令）与字段说明（Open Targets 的关联打分与逐条证据、PubMed 的文献与摘要），由 `bash` 跑；循环层决定读哪份说明、跑哪个脚本、跑几次；核验层是技能自带的一个脚本，扫本轮落盘的原始响应建台账；模型层只负责判断。答案是「工具结果的再组织」，不是模型的记忆。证据链完整形式是：`疾病 id → 基因 id → 关联 score + 证据类型 → 具体 PMID`，每一段都是数据库里能打开的记录。

**Q：工具使用是否提升了答案的可靠性？**

是，但可靠性要拆成三件事分别看：

| 维度 | 有工具 | 无工具 |
|---|---|---|
| 引用可核验性 | 三题引用 78 个标识符，未核验 0 个（覆盖率 1.0） | 一个标识符都没有，无从核验 |
| 数值可复算 | 分数、计数、排行都能回数据库复算 | 只有「OR 约 3–4」这类无法定位来源的区间 |
| 缺口可定位 | 能说清「哪条没取逐条证据」 | 只有「还有缺失遗传力」这类笼统说法 |

同时它没有提升的部分要说清楚：工具不能扩大数据源覆盖面（TB 标志物案例），也不能防止「引用真实但解读过头」。所以「有工具」把答案从「读起来可信」变成「可以被程序检查」，把判断从读感变成可核验的指标——这是可靠性的实际来源。

**局限**

1. 单数据源主导：三题的正确性依赖 Open Targets 一家；它的关联打分是证据加权，不等于临床因果，agent 在 Q2 里自己点出了这一点，但没有一种工具能给它「突变频率」「诊断性能」这类数据。
2. `coverage` 只证明「引用在工具输出里出现过」，不证明「结论正确」；语义层的正确性仍要人来判。反过来，台账也不是防伪的：扫的是落盘的原始响应，在 `bash` 里 `echo` 一个 id 只进终端不进台账，但用 `write` 把它们存进 `.tmp/responses/` 同样能被扫到。
3. 上下文不会压缩：`context.messages` 只增不减，长会话迟早超出模型窗口（照 pi 的 compaction 章节移植即可补）。
4. 轮次预算与收束只解决「半成品被当成成品」，不解决「答案拼得对不对」；失败案例里 A 与 C 的差别只在于剩余轮次够不够把话说究。

**改进方向**：在技能正文里补「能力边界」一段让模型早停；预算耗尽改成「按已有证据分级作答」；对引用做二次抽检（取 PMID 摘要与原句比对）。另有一条通用工具带来的代价：模型每题要先花 2–4 次调用读技能说明，这部分开销可以用「主线技能放进系统提示词全文」消掉。

## 10. 现场演示：可连续对话的 web 界面

notebook 是静态报告，现场问答用 `server.py` + 浏览器页面：后端还是 §3 那个通用 `Agent`，只把 loop 的事件实时推给浏览器，所以演示里看到的工具调用就是 §6 统计的那些调用，走的是同一条代码路径。前端本身不持能力，只把事件流渲染成对话。启动一条命令：

```bash
./run_server.sh          # http://localhost:8765
```

对报告有意义的只有三件事。一、会话是连续对话：同一个浏览器会话对应一个 agent 对象，`context.messages` 不复位，可以接着上一轮追问，和 §6.2 是同一个机制。二、过程可见：每轮的工具调用与调用次数在页面上实时显示，答案下方的引用可以点回 PubMed、Ensembl、Open Targets，也就是 §5 台账里那些标识符的另一种打开方式。三、边界：落盘的只有问答记录，模型侧上下文不存盘，重启进程后旧会话只能查看、不能接着追问。

文件分工只有两处：`server.py` 是后端接口（读模型与提示词、会话的建/选/改名/删、流式问答、中止；会话记录落盘到 `.tmp/sessions/<id>.json`，核验仍调 `skills/evidence-discipline/scripts/verify.py`），`web/` 是页面本体（源码在 `web/src/`，`npm run build` 的产物 `web/dist/` 由 `server.py` 直接发）；`run_server.sh` 后台启动并把 pid 写到 `.tmp/server.pid`，重复执行不会起两份。

演示不另做评测：界面上能核对的，就是 §6 的调用序列与 §5 的核验数字。

## 11. 怎么重跑

```bash
cd tasks/learning/biomedical-agent-20261118
# 只需一次：建 venv 与 kernel
python3 -m venv ~/.venvs/bioagent
~/.venvs/bioagent/bin/pip install nbformat nbclient nbconvert ipykernel requests pyyaml
~/.venvs/bioagent/bin/python -m ipykernel install --prefix ~/.venvs/bioagent --name bioagent
# 执行报告
~/.venvs/bioagent/bin/python -m nbconvert --execute --to notebook --inplace report.ipynb \
  --ExecutePreprocessor.timeout=1800 \
  --ExecutePreprocessor.kernel_name=bioagent
```

模型网关与凭据默认从 `~/.pi/agent/models.json` 的 `magpie` provider 读，可用 `BIOAGENT_BASE_URL` / `BIOAGENT_API_KEY` 覆盖。换模型只改 `MODEL` 常量。核验脚本只用标准库，不额外装包。代码风格检查：

```bash
~/.venvs/bioagent/bin/ruff check .        # PEP 8
```

整份报告实跑约 12 分钟（3 题 + 多轮 + 基线 + 失败案例三组）。